In [6]:
import json
with open("tasks_mcq.json", "r") as f:
    tasks = json.load(f)


In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
from urllib.parse import quote
import random
from time import sleep
def get_isomers(smiles):
    try:
        # Convert SMILES to RDKit molecule object
        mol = Chem.MolFromSmiles(smiles)
        
        if mol is None:
            return "Invalid SMILES string"
        
        # Get the molecular formula
        formula = Descriptors.MolecularFormula(mol)
        
    except Exception as e:
        raise e
    
    url = [
        "/compound/fastformula/",
        "/cids/JSON",
    ]
    base_url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

    url = base_url + url[0] + quote(str(formula)) + url[1]
    isomers_cids = (get_data_from_url(url))["IdentifierList"]["CID"]
    random.shuffle(isomers_cids)
    # Limit the number of isomers to process
    isomers_cids = isomers_cids[:3]
    data = []
    for i in isomers_cids:
        sleep(0.5)
        cid = i
        try:
            # Isomeric SMILES to capture enantiomers
            smiles = _get_isomeric_smiles()
            if smiles:
                data.append(smiles)
    

In [18]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

def get_molecular_formula(smiles):
    """
    Convert a SMILES string to molecular formula.
    
    Parameters:
    smiles (str): The SMILES representation of the molecule
    
    Returns:
    str: The molecular formula (e.g., C6H6O)
    """
    try:
        # Convert SMILES to RDKit molecule object
        mol = Chem.MolFromSmiles(smiles)
        
        if mol is None:
            return "Invalid SMILES string"
        
        # Get the molecular formula
        formula = rdMolDescriptors.CalcMolFormula(mol)
        
        return formula
    
    except Exception as e:
        return f"Error: {str(e)}"

In [20]:
import json
import random

with open("tasks_mcq.json", "r") as f:
    tasks = json.load(f)

import modal

get_isomers = modal.Function.from_name("chemenv", "get_compound_isomers_pubchem_by_formula")

for task in tasks:
    description = task["spectra"]
    smiles = task["smiles"]
    formula = get_molecular_formula(smiles)
    print(formula)
    isomers = get_isomers.remote(formula)
    
    # Take 3 random isomers from the list
    random_isomers = random.sample(isomers, 3)
    
    # Add the original SMILES to the options
    options = [smiles] + random_isomers
    
    # Shuffle the options
    random.shuffle(options)
    
    # Create options string in format "A. option1, B. option2, C. option3, D. option4"
    option_letters = ["A", "B", "C", "D"]
    options_text = ", ".join([f"{letter}. {option}" for letter, option in zip(option_letters, options)])
    
    # Update the task description
    task["spectra"] = f"{description}\n\nThe options are: {options_text}"
    
    # Store which option (A, B, C, D) contains the original SMILES
    correct_index = options.index(smiles)
    task["correct_option"] = option_letters[correct_index]

# Optionally, save the updated tasks back to the file
with open("tasks_mcq_updated.json", "w") as f:
    json.dump(tasks, f, indent=4)



C21H20N4O8S2
C14H13ClN2O2
C8H16O2Si
C18H14O2
C18H24N2
C9H8O4
C12H15Cl
C5H8O2
C13H10O
C16H16O
C8H9BF3K
C10H14
C3H5N3O2
C7H7NO
C9H8O4
